# Exercise 2 — Predict Miami house prices with TabPFN-3.5 Plus

Use TabPFN through **SAP AI Core**, then compare its predictions with a training-median baseline and XGBoost. The Miami housing dataset contains 13,776 property sales and 15 features describing location, size, age, and nearby amenities. You can switch to your own regression dataset in the configuration cell below.

TabPFN learns from labelled **context rows** supplied with each request; query rows contain only features. Held-out sale prices stay local for evaluation. This notebook requires a running SAP TabPFN-3.5 Plus deployment for prediction. See the [setup and data attribution](README.md) and [SAP API documentation](https://help.sap.com/docs/sap-ai-core/generative-ai/tabpfn-3-5-plus).


## 1. Set up

Launch this notebook from `exercises/ex2`. You should have filled in the SAP credentials and deployment IDs into the `.env` file.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

from helpers import (
    baseline_predictions,
    download_miami_housing,
    predict_sap_tabpfn,
    predict_sap_tabpfn_quantiles,
    predict_sap_tabpfn_full,
    plot_predictive_distribution,
    explain_sap_tabpfn,
    plot_shap_explanation,
    plot_miami_shap_explanation,
    regression_metrics,
    select_context,
    split_regression_data,
)

load_dotenv()


## 2. Choose your data and prediction target

`SALE_PRC` is the property sale price in US dollars. The supplied Miami dataset has already had the parcel identifier removed, so `DROP_COLUMNS` is empty. Use only information available when valuing a home; for a real deployment, check when each appraisal-derived feature was recorded.

The helper downloads [Miami housing from OpenML](https://www.openml.org/search?type=data&id=46942) on the first run and caches it as `miami_housing.csv`. See the README for attribution and the CC BY-NC-SA source license.

**Your own CSV:** change `CSV_PATH`, `TARGET_COLUMN`, and `DROP_COLUMNS`, then run the remaining cells. Set `SORT_COLUMN` for an ordered time split; leave it `None` for independent rows. Parse date strings explicitly below when using a time split. This exercise supports numeric regression targets; class labels need a classifier and classification metrics.


In [ ]:
CSV_PATH = Path("miami_housing.csv")
TARGET_COLUMN = "SALE_PRC"
DROP_COLUMNS = []  # add identifiers or leaked outcomes for your own data
SORT_COLUMN = None  # e.g. "sale_date" for your own time-ordered data
TEST_SIZE = 0.2
MAX_CONTEXT_ROWS = 2_048  # workshop choice, not the model's maximum
RANDOM_STATE = 42

# For another dataset, for example:
# CSV_PATH = Path("my_data.csv")
# TARGET_COLUMN = "sales"
# DROP_COLUMNS = ["record_id", "revenue_after_sale"]

if CSV_PATH == Path("miami_housing.csv") and not CSV_PATH.exists():
    download_miami_housing(CSV_PATH)
df = pd.read_csv(CSV_PATH)
if SORT_COLUMN is not None:
    df[SORT_COLUMN] = pd.to_datetime(df[SORT_COLUMN], errors="raise")
print(f"{len(df):,} rows, {len(df.columns)} columns")
display(df.head())


## 3. Hold out evaluation rows and select context

The split happens before any baseline preprocessing. Missing targets cannot be used as labels; the helper reports and removes them. Non-numeric or infinite targets raise an error so you can fix them. Feature missing values are retained. For a time split, the test set is the newest rows and the context is the latest training rows. For repeated customers or other groups, replace this split with a group-aware split before drawing conclusions.


In [ ]:
X_train, X_test, y_train, y_test = split_regression_data(
    df, target=TARGET_COLUMN, drop_columns=DROP_COLUMNS,
    test_size=TEST_SIZE, random_state=RANDOM_STATE, sort_column=SORT_COLUMN,
)
X_context, y_context = select_context(
    X_train, y_train, max_rows=MAX_CONTEXT_ROWS,
    random_state=RANDOM_STATE, chronological=SORT_COLUMN is not None,
)
print(f"Context: {len(X_context):,} | Test: {len(X_test):,} | Features: {X_context.shape[1]}")
display(X_context.head())


## 4. Establish baselines

All three methods see exactly the same context and test rows. The median predicts a single training-set value. XGBoost fits a fixed, untuned tree ensemble; its numeric imputation and categorical encoding are learned only from context rows. TabPFN receives the original feature values, without one-hot encoding.


In [ ]:
predictions = baseline_predictions(
    X_context, y_context, X_test, random_state=RANDOM_STATE,
)
display(regression_metrics(y_test, predictions).round(2))


## 5. Predict with the SAP deployment

The helper authenticates with SAP AI Core, converts context and query data into the documented JSON format, sends requests in batches, and returns predictions in test-row order. Unlike RPT, the TabPFN `/predict` endpoint takes separate `x_train`, `y_train`, `x_test`, and `task_config` fields; its response uses `prediction`. No Prior Labs client token is needed here. This cell is the point at which context and test features are sent to your SAP deployment; use data suitable for that environment.


In [ ]:
predictions["TabPFN-3.5 Plus"] = predict_sap_tabpfn(
    X_context, y_context, X_test, target_name=TARGET_COLUMN, random_state=RANDOM_STATE,
)


## 6. Compare and inspect errors

Lower is better for every metric: **MAE** is the average absolute error in target units; **RMSE** puts more weight on large misses; **MAPE (%)** is the average absolute percentage error. MAPE excludes zero-valued targets (and is undefined if all targets are zero); near-zero targets can still dominate it. For Miami housing, MAE and RMSE are in US dollars.


In [ ]:
scores = regression_metrics(y_test, predictions)
display(scores.round(2))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for metric, ax in zip(scores.columns, axes):
    scores[metric].plot.bar(ax=ax, title=metric, rot=25)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


In [ ]:
results = X_test.copy()
results["Actual"] = y_test.to_numpy()
for name, values in predictions.items():
    results[f"Prediction: {name}"] = values
results["Absolute error: TabPFN"] = (
    results["Actual"] - results["Prediction: TabPFN-3.5 Plus"]
).abs()
display(results.sort_values("Absolute error: TabPFN", ascending=False).head(10))


## 7. Quantiles and the predictive distribution

A point estimate hides uncertainty. Request the 10th, 50th and 90th percentiles with `output_type="quantiles"`. The interval from the 10th to the 90th percentile is a **central 80% predictive interval**, not a confidence interval for the mean or a guarantee of coverage. We use the first 20 test homes to keep this extra request small.


In [ ]:
X_example = X_test.iloc[:20]
quantiles = predict_sap_tabpfn_quantiles(
    X_context, y_context, X_example, target_name=TARGET_COLUMN,
    quantiles=(0.1, 0.5, 0.9), random_state=RANDOM_STATE,
)
display(quantiles.rename(columns={0.1: "10%", 0.5: "50%", 0.9: "90%"}))

fig, ax = plt.subplots(figsize=(9, 3.5))
positions = range(len(X_example))
ax.fill_between(positions, quantiles[0.1], quantiles[0.9], alpha=0.25, label="80% predictive interval")
ax.plot(positions, quantiles[0.5], "o", label="Predicted median")
ax.plot(positions, y_test.loc[X_example.index], "x", label="Actual")
ax.set(xlabel="Test example", ylabel=TARGET_COLUMN, title="Predicted quantiles")
ax.legend()
plt.tight_layout()
plt.show()


Use `output_type="full"` to retrieve the distribution, rather than just selected quantiles. The helper below makes that request for **one home** and plots the returned probability mass in each bin. The outer bins include the distribution tails; bar heights are probabilities, not a continuous density. The horizontal axis zooms to the central 99% of predicted mass so extreme tail boundaries do not hide the distribution. The actual outcome is shown for comparison. Full-output requests are limited to 400 query rows by SAP.


In [ ]:
full_output = predict_sap_tabpfn_full(
    X_context, y_context, X_example.iloc[:1], target_name=TARGET_COLUMN,
    random_state=RANDOM_STATE,
)
print("Full-output fields:", sorted(full_output))
ax = plot_predictive_distribution(full_output, row=0)
ax.axvline(y_test.loc[X_example.index[0]], color="black", linestyle="--", label="Actual")
ax.set(xlabel=TARGET_COLUMN, title="Predictive distribution for one test example")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Explain individual house-price predictions with SHAP

Each bar shows how a feature contributes to **this home's predicted price** relative to the average model prediction for 32 reference homes. **Blue / left lowers** the prediction; **red / right raises** it. The labels state the dollar contribution directly; **k = $1,000**. The summary at the top connects the reference prediction, total contribution and predicted price.

For Miami housing, feature names and units are spelled out. Latitude and longitude say whether the home is **north/south or east/west of the reference median**; distance features say whether it is closer or farther. These descriptions use the same reference homes as SHAP. Water distance and latitude always remain visible. Feature medians provide context; SHAP contributions are not computed simply by subtracting those medians.

Each cell below explains one house in **one SAP prediction request**. Change `row_position` to inspect another home. The background and labelled context stay fixed. The separate `plot_miami_shap_explanation` helper handles presentation; other datasets retain the standard waterfall.

These approximate attributions describe the model, not causal effects or general rules about moving north or closer to water. Correlated features can make contributions sensitive to the background and permutation budget. The helper requires finite numeric features.


In [ ]:
# Explain one held-out house; choose a different position to try another.
row_position = 18
house = X_test.iloc[[row_position]]
shap_values, shap_details = explain_sap_tabpfn(
    X_context, y_context, house, target_name=TARGET_COLUMN,
    background_size=32, n_permutations=32, random_state=RANDOM_STATE,
)
display(pd.DataFrame({
    "Actual price": y_test.loc[house.index],
    "Predicted price": shap_details["predictions"],
}).round(0))
if CSV_PATH.name == "miami_housing.csv" and TARGET_COLUMN == "SALE_PRC":
    shap_background = X_context.sample(
        n=shap_details["background_rows"], random_state=RANDOM_STATE,
    )
    plot_miami_shap_explanation(
        shap_values, shap_background,
        prediction=shap_details["predictions"][0], actual=y_test.loc[house.index[0]],
        row_position=row_position, source_row=house.index[0],
    )
else:
    plot_shap_explanation(shap_values, target_name=TARGET_COLUMN)
plt.show()


In [ ]:
# Explain one held-out house; choose a different position to try another.
row_position = 4
house = X_test.iloc[[row_position]]
shap_values, shap_details = explain_sap_tabpfn(
    X_context, y_context, house, target_name=TARGET_COLUMN,
    background_size=32, n_permutations=32, random_state=RANDOM_STATE,
)
display(pd.DataFrame({
    "Actual price": y_test.loc[house.index],
    "Predicted price": shap_details["predictions"],
}).round(0))
if CSV_PATH.name == "miami_housing.csv" and TARGET_COLUMN == "SALE_PRC":
    shap_background = X_context.sample(
        n=shap_details["background_rows"], random_state=RANDOM_STATE,
    )
    plot_miami_shap_explanation(
        shap_values, shap_background,
        prediction=shap_details["predictions"][0], actual=y_test.loc[house.index[0]],
        row_position=row_position, source_row=house.index[0],
    )
else:
    plot_shap_explanation(shap_values, target_name=TARGET_COLUMN)
plt.show()


## 9. Try your own data and iterate

Return to the configuration cell, point to your CSV, choose a numeric target and remove identifiers or leaked outcomes. Re-run from that cell to compare all three methods automatically. For a date-based split, choose the date that reflects when the prediction would actually be made.

- Does TabPFN beat both baselines on MAE and MAPE? Which metric matters for your use case?
- Which examples have the largest errors? Are their feature values unlike the context rows?
- Try more context rows, keeping the test split fixed and staying within your deployment's documented limits. All methods will use the new context.

Keep the test set for evaluation; use a separate validation split if you tune model settings repeatedly.


## 10. Optional: classify phishing websites

Now predict a **category** instead of a price. The website phishing dataset has 1,353 examples, nine features and three labels: **Legitimate**, **Phishy**, and **Suspicious**. We compare TabPFN with a majority-class baseline and XGBoost.

The data contains repeated feature patterns. Our repeatable split keeps identical feature vectors together, so they cannot appear in both training and test data. The test row fraction is approximate because groups differ in size.

Source: [OpenML 46963](https://www.openml.org/search?type=data&id=46963), derived from [UCI Website Phishing](https://doi.org/10.24432/C5B301), CC BY 4.0. See the README for attribution.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from helpers import (
    download_website_phishing, split_classification_data, select_classification_context,
    classification_baseline_predictions, predict_sap_tabpfn_classification,
    classification_metrics,
)

CLASSIFICATION_CSV = Path("website_phishing.csv")
CLASSIFICATION_TARGET = "WebsiteType"
CLASSIFICATION_DROP_COLUMNS = []
CLASSIFICATION_CONTEXT_ROWS = 2048

if CLASSIFICATION_CSV == Path("website_phishing.csv") and not CLASSIFICATION_CSV.exists():
    download_website_phishing(CLASSIFICATION_CSV)
classification_df = pd.read_csv(CLASSIFICATION_CSV)
display(classification_df.head())
display(classification_df[CLASSIFICATION_TARGET].value_counts().rename("Examples"))

In [ ]:
clf_X_train, clf_X_test, clf_y_train, clf_y_test = split_classification_data(
    classification_df, CLASSIFICATION_TARGET, drop_columns=CLASSIFICATION_DROP_COLUMNS,
    test_size=0.2, random_state=RANDOM_STATE, group_identical_features=True,
)
clf_X_context, clf_y_context = select_classification_context(
    clf_X_train, clf_y_train, max_rows=CLASSIFICATION_CONTEXT_ROWS, random_state=RANDOM_STATE,
)
print(f"Context: {len(clf_X_context):,} rows; test: {len(clf_X_test):,} rows")
display(pd.DataFrame({"Context": clf_y_context.value_counts(), "Test": clf_y_test.value_counts()}))

### Compare the three classifiers

The helper sends `task="classification"` and `output_type="preds"` to the same SAP deployment. All models use the same context and held-out examples. A median has no meaning for these labels, so the naive baseline predicts the most frequent context class.

In [ ]:
clf_predictions = classification_baseline_predictions(
    clf_X_context, clf_y_context, clf_X_test, random_state=RANDOM_STATE,
)
clf_predictions["TabPFN-3.5 Plus"] = predict_sap_tabpfn_classification(
    clf_X_context, clf_y_context, clf_X_test,
    target_name=CLASSIFICATION_TARGET, random_state=RANDOM_STATE,
)
display(classification_metrics(clf_y_test, clf_predictions).round(3))

Higher scores are better. **Accuracy** counts correct predictions. **Balanced accuracy** averages recall across classes, so the smaller Suspicious class matters equally. **Macro F1** averages each class's F1 score. The confusion matrix shows which categories TabPFN confuses.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    clf_y_test, clf_predictions["TabPFN-3.5 Plus"],
    labels=sorted(clf_y_context.unique()), cmap="Blues", colorbar=False,
)
plt.title("TabPFN: website classification")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "Actual": clf_y_test,
    **{name: values for name, values in clf_predictions.items()},
}).head(10))

### Try your own classification dataset

Change `CLASSIFICATION_CSV`, `CLASSIFICATION_TARGET`, and `CLASSIFICATION_DROP_COLUMNS`, then rerun this section. Use a categorical target with at least two classes and enough examples of every class for both splits. Drop identifiers and columns that reveal the answer. Missing targets are removed; feature preprocessing is handled by the helpers.

Keep identical-feature grouping enabled for repeated patterns. If rows repeat customers or other entities, use an entity-aware split; use a chronological split for future predictions. For independent rows, `group_identical_features=False` uses a stratified random split. You can change `CLASSIFICATION_CONTEXT_ROWS` to explore context size, but use a validation set when tuning.

Try inspecting misclassified Suspicious websites or dropping a feature. Which class is hardest to recognize?

Continue to [Exercise 3 — Tabular AI orchestration](../ex3/README.md).